# CN Variable Analysis

**Two parts:**
1. Distribution statistics for all CN variables
2. Monthly mean maps (June / July / August) showing spatial cumulation of network patterns

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import xarray as xr

%matplotlib inline
plt.rcParams['figure.dpi'] = 120


In [ ]:
DATA_FILE = "/Users/malgorzatazdych/Documents/Master2/Thesis/Dataset/MasterThesis/full_processed_training_dataset.nc"
NUTS_FILE = "/Users/malgorzatazdych/Documents/Master2/Thesis/Dataset/MasterThesis/NUTS_RG_01M_2024_4326_LEVL_2.geojson"
OUT_DIR   = "cn_analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

CN_VARS = ["CC", "BC", "DC", "ID", "OD", "CC_target_next_day"]

CMAPS = {
    "CC":                 "YlOrRd",
    "BC":                 "PuRd",
    "DC":                 "Blues",
    "ID":                 "Greens",
    "OD":                 "Oranges",
    "CC_target_next_day": "YlOrRd",
}

MONTHS      = [6, 7, 8]
MONTH_NAMES = {6: "June", 7: "July", 8: "August"}


## Load Dataset

In [5]:
print("Loading dataset...")
ds    = xr.open_dataset(DATA_FILE)
lats  = ds["lat"].values
lons  = ds["lon"].values
times = pd.DatetimeIndex(ds["time"].values)

print(f"  Grid: {len(lats)} lat x {len(lons)} lon  |  {len(times)} timesteps")
print(f"  Period: {times[0].date()} to {times[-1].date()}")
print(f"  Variables: {list(ds.data_vars)}")

# Land mask (south-first)
land_mask_sf = ds["land_mask"].isel(time=0).values.astype(bool)

# North-first for plotting
lats_nf = lats[::-1]
LON2D, LAT2D = np.meshgrid(lons, lats_nf)
land_mask_nf  = land_mask_sf[::-1, :]

GEO_ASPECT = 1.0 / np.cos(np.radians((lats.min() + lats.max()) / 2))

def setup_ax(ax):
    ax.set_aspect(GEO_ASPECT)
    ax.set_xlim(lons.min(), lons.max())
    ax.set_ylim(lats.min(), lats.max())


Loading dataset...
  Grid: 141 lat x 264 lon  |  2667 timesteps
  Period: 1990-06-01 to 2020-08-30
  Variables: ['DC', 'CC', 'BC', 'ID', 'OD', 'is_heatwave', 'swvl1', 'land_mask', 'u', 'v', 'z', 'divergence', 'CC_target_next_day', 'event_label']


In [6]:
nuts    = gpd.read_file(NUTS_FILE).to_crs("EPSG:4326")
nuts_eu = nuts.cx[lons.min():lons.max(), lats.min():lats.max()]
print(f"  NUTS features: {len(nuts_eu)}")


DataSourceError: NUTS_RG_01M_2024_4326_LEVL_2.geojson: No such file or directory

---
## Part 1 — Distribution Statistics

For each CN variable: range, quantiles, zero-inflation, unique value count.  
Crucial for understanding whether regression makes sense and what threshold choices are appropriate.

In [ ]:
stats_rows = []
dist_data  = {}

for var in CN_VARS:
    arr      = ds[var].values                     # (T, lat, lon) south-first
    arr_land = arr[:, land_mask_sf].ravel()
    arr_land = arr_land[~np.isnan(arr_land)]

    frac_zero = (arr_land == 0).mean()
    frac_one  = (arr_land == 1).mean()
    n_unique  = len(np.unique(arr_land))

    row = {
        "variable":  var,
        "n_values":  len(arr_land),
        "min":       arr_land.min(),
        "max":       arr_land.max(),
        "mean":      arr_land.mean(),
        "median":    np.median(arr_land),
        "std":       arr_land.std(),
        "q10":       np.quantile(arr_land, 0.10),
        "q25":       np.quantile(arr_land, 0.25),
        "q40":       np.quantile(arr_land, 0.40),
        "q70":       np.quantile(arr_land, 0.70),
        "q90":       np.quantile(arr_land, 0.90),
        "q99":       np.quantile(arr_land, 0.99),
        "frac_zero": frac_zero,
        "frac_one":  frac_one,
        "n_unique":  n_unique,
    }
    stats_rows.append(row)
    dist_data[var] = arr_land

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(os.path.join(OUT_DIR, "cn_distribution_stats.csv"), index=False)
stats_df


### Interpretation guide

| Variable | Expected behaviour | Regression sensibility |
|---|---|---|
| **CC** | High zero-inflation (94%); near-binary | Regression on sparse binary — smooth continuous output is a spatial probability |
| **BC** | Moderate zero-inflation; right-skewed | Regression viable — more continuous than CC |
| **DC** | Bimodal (Julia thesis Fig 1) | Regression viable |
| **ID** | Right-skewed tail | Regression viable; log1p transform helps |
| **OD** | Right-skewed tail | Same as ID |
| **CC_target_next_day** | Same as CC (it is shifted CC) | Regression target — model learns spatial activation score |


### Fig D1 — Distribution histograms (all variables)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, var in enumerate(CN_VARS):
    ax   = axes[i]
    vals = dist_data[var]
    p99  = np.percentile(vals, 99)

    ax.hist(vals[vals <= p99], bins=120, color=plt.cm.tab10(i),
            alpha=0.75, edgecolor="none", density=True)

    for q_val, q_label, col in [
        (np.quantile(vals, 0.40), "q40", "#e67e22"),
        (np.quantile(vals, 0.70), "q70", "#c0392b"),
    ]:
        ax.axvline(q_val, color=col, linewidth=1.8, linestyle="--",
                   label=f"{q_label}={q_val:.3f}")

    r = stats_df[stats_df.variable == var].iloc[0]
    ax.set_title(f"{var}\n"
                 f"zeros={r.frac_zero*100:.1f}%  unique={int(r.n_unique):,}",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Value (clipped to p99)", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

fig.suptitle("CN Variable Distributions — all years, land pixels (clipped to p99)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figD1_cn_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved figD1")


### Fig D2 — Zero-inflation comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
frac_z  = stats_df["frac_zero"].values * 100
colors  = [plt.cm.tab10(i) for i in range(len(CN_VARS))]
bars    = ax.bar(stats_df["variable"], frac_z, color=colors, alpha=0.85, edgecolor="white")

for bar, val in zip(bars, frac_z):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("% of land-pixel-days == 0", fontsize=11)
ax.set_title("Zero-inflation by CN variable (all years, land pixels)", fontsize=11)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figD2_zero_inflation.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved figD2")


---
## Part 2 — Monthly Mean Maps (June / July / August)

For each CN variable: mean spatial map separately for June, July, August  
averaged across all years 1990–2020.

This shows the **cumulative seasonal pattern** — where network activity concentrates  
during each month of summer, revealing how heatwave network structure evolves  
from early season (June) to peak (July/August).


In [ ]:
for var in CN_VARS:
    print(f"  {var}...")
    arr = ds[var].values   # (T, lat, lon) south-first

    monthly_means = {}
    for m in MONTHS:
        mask_m = (times.month == m)
        mean_m = np.nanmean(arr[mask_m], axis=0)
        mean_m[~land_mask_sf] = np.nan
        monthly_means[m] = mean_m[::-1, :]   # flip to N-first

    all_vals = np.concatenate([v[~np.isnan(v)].ravel() for v in monthly_means.values()])
    vmin = float(np.nanpercentile(all_vals, 2))
    vmax = float(np.nanpercentile(all_vals, 98))
    cmap = CMAPS.get(var, "viridis")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for col, m in enumerate(MONTHS):
        ax = axes[col]
        pc = ax.pcolormesh(LON2D, LAT2D, monthly_means[m],
                           cmap=cmap, vmin=vmin, vmax=vmax,
                           shading="nearest", zorder=1)
        nuts_eu.boundary.plot(ax=ax, linewidth=0.35,
                              color="black", alpha=0.55, zorder=2)
        setup_ax(ax)

        ax.set_xticks(np.linspace(lons.min(), lons.max(), 4))
        ax.set_xticklabels(
            [f"{v:.0f}E" if v >= 0 else f"{abs(v):.0f}W"
             for v in np.linspace(lons.min(), lons.max(), 4)], fontsize=7)
        ax.set_yticks(np.linspace(lats.min(), lats.max(), 4))
        ax.set_yticklabels([f"{v:.0f}N"
             for v in np.linspace(lats.min(), lats.max(), 4)], fontsize=7)

        vals_m = monthly_means[m][~np.isnan(monthly_means[m])].ravel()
        frac_nz = (vals_m > 0).mean()
        ax.set_title(f"{MONTH_NAMES[m]}\n"
                     f"mean={vals_m.mean():.4f}  non-zero={frac_nz*100:.1f}%",
                     fontsize=9)

    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.12, 0.018, 0.75])
    sm = plt.cm.ScalarMappable(cmap=cmap,
                               norm=mcolors.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label=f"Mean {var}")

    fig.suptitle(f"{var} — Monthly mean (JJA, 1990–2020)\n"
                 f"Spatial cumulation of network activity across summer",
                 fontsize=11, fontweight="bold")

    path = os.path.join(OUT_DIR, f"figM_{var}_monthly.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"    Saved {path}")


### Fig M7 — All variables, August (combined overview)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

for i, var in enumerate(CN_VARS):
    ax  = axes[i]
    arr = ds[var].values
    mean_aug = np.nanmean(arr[times.month == 8], axis=0)
    mean_aug[~land_mask_sf] = np.nan
    mean_aug_nf = mean_aug[::-1, :]

    vals = mean_aug_nf[~np.isnan(mean_aug_nf)].ravel()
    vmin = float(np.nanpercentile(vals, 2))
    vmax = float(np.nanpercentile(vals, 98))
    cmap = CMAPS.get(var, "viridis")

    pc = ax.pcolormesh(LON2D, LAT2D, mean_aug_nf,
                       cmap=cmap, vmin=vmin, vmax=vmax,
                       shading="nearest", zorder=1)
    nuts_eu.boundary.plot(ax=ax, linewidth=0.3,
                          color="black", alpha=0.5, zorder=2)
    plt.colorbar(pc, ax=ax, fraction=0.04, label=var)
    setup_ax(ax)
    ax.set_title(f"{var} — August mean", fontsize=10, fontweight="bold")
    ax.tick_params(labelsize=6)

fig.suptitle("All CN variables — August mean (1990–2020)\n"
             "Spatial patterns of network coefficients during peak heatwave season",
             fontsize=12, fontweight="bold")
plt.tight_layout()
path = os.path.join(OUT_DIR, "figM_ALL_august.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {path}")


---
## Cleanup

In [ ]:
ds.close()
print(f"All outputs saved to: {OUT_DIR}/")
print("Done.")
